# Model Experiments

Offline model comparison for company performance prediction using committed snapshots and MLflow tracking.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.features.build_features import build_features
from src.models.train import train_models, year_based_split
from src.utils import load_config

In [ ]:
config = load_config(PROJECT_ROOT / "config.yaml")
outputs = build_features(config)

features = pd.read_csv(outputs["features"])
test_year = int(config["training"]["test_year"])
train_df, test_df = year_based_split(features, test_year)

print(f"test_year: {test_year}")
print(f"train rows: {len(train_df)} ({train_df['fiscal_year'].min()}-{train_df['fiscal_year'].max()})")
print(f"test rows: {len(test_df)} (year {test_df['fiscal_year'].unique()[0]})")

In [ ]:
artifacts = train_models(config)

import json

metadata = json.loads((PROJECT_ROOT / "models" / "metadata.json").read_text(encoding="utf-8"))

rows = []
for model_name, results in metadata["models"].items():
    row = {"model": model_name, **results["test_metrics"]}
    rows.append(row)

comparison = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
comparison

## MLflow UI

From the project root, start the MLflow UI to inspect logged runs:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open http://127.0.0.1:5000 and select the `company-performance-prediction` experiment.